# Подготовка данных к работе

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_parquet("../data/processed/filtered.parquet")

## Удаление признаков

По проведенного разведочного анализа было решено удалить следующие константные признаки:
- `text__obscene_proportion`, 
- `punct__mean_usage_quote`,
- `punct__mean_usage_ampersand`, 
- `entities__mean_usage_URL`,
- `punct__mean_usage_closebracket`, 
- `punct__mean_usage_openbracket`,
- `punct__mean_usage_semicolon`, 
- `punct__mean_usage_lessthan`,
- `punct__mean_usage_greaterthan`, 
- `pos__mean_usage_INTJ`,
- `punct__mean_usage_slash`, 
- `punct__mean_usage_asterisk`,
- `punct__mean_usage_underscore`, 
- `punct__mean_usage_tilde`,
- `entities__mean_usage_TAG`, 
- `entities__mean_usage_EMAIL`,
- `entities__mean_usage_PHONE`, 
- `entities__mean_usage_POINTER`

И неиспользуемые признаки:
- `user_id`
- `source_url`
- `text`

In [3]:
df.drop(columns=[
    'text__obscene_proportion', 
    'punct__mean_usage_quote',
    'punct__mean_usage_ampersand', 
    'entities__mean_usage_URL',
    'punct__mean_usage_closebracket', 
    'punct__mean_usage_openbracket',
    'punct__mean_usage_semicolon', 
    'punct__mean_usage_lessthan',
    'punct__mean_usage_greaterthan', 
    'pos__mean_usage_INTJ',
    'punct__mean_usage_slash', 
    'punct__mean_usage_asterisk',
    'punct__mean_usage_underscore', 
    'punct__mean_usage_tilde',
    'entities__mean_usage_TAG', 
    'entities__mean_usage_EMAIL',
    'entities__mean_usage_PHONE', 
    'entities__mean_usage_POINTER',
    'user_id',
    'source_url',
    'text'
], inplace=True)

## Флаг использования

Следующие признаки интенсивности использования соответствующей метрики следует дополнить бинарным флагом использования:
- `pos__pair_of_adv_per_sentence`,
- `pos__mean_usage_AUX`,
- `pos__mean_usage_NUM`,
- `pos__PROPN_PUNCT`,
- `pos__PUNCT_PRON`,
- `punct__mean_usage_colon`,
- `punct__mean_usage_question`,
- `punct__mean_usage_openparen`,
- `punct__mean_usage_closeparen`,
- `punct__mean_usage_hyphen`,
- `punct__mean_usage_exclamation`,
- `entities__mean_usage_NUMBER`,
- `entities__mean_usage_QUOTE`,
- `entities__mean_usage_PUNCEM`,
- `entities__mean_usage_ADDRESS`

In [4]:
combo_features = { 
    "pos__pair_of_adv_per_sentence",
    "pos__mean_usage_AUX",
    "pos__mean_usage_NUM",
    "pos__PROPN_PUNCT",
    "pos__PUNCT_PRON",
    "punct__mean_usage_colon",
    "punct__mean_usage_question",
    "punct__mean_usage_openparen",
    "punct__mean_usage_closeparen",
    "punct__mean_usage_hyphen",
    "punct__mean_usage_exclamation",
    "entities__mean_usage_NUMBER",
    "entities__mean_usage_QUOTE",
    "entities__mean_usage_PUNCEM",
    "entities__mean_usage_ADDRESS",
}
for col in combo_features:
    df[f'{col}_used'] = (df[col] > 0).astype(int)

## Бинаризация возраста
В качестве порога "взрослости" выбрано значение 35 лет. `<= 35` - молодой, `> 35` - взрослый

In [5]:
df['age'] = (df['age'] <= 35).astype(int)

## Кодирование признака пола

In [6]:
df['gender'] = df['gender'].map({"мужской": 1, "женский": 0})

## Разбиение на тестовую и обучающую выборку
Для сохранения баланса классов будем использовать многомерную стратифиикацию. В качестве пропорции используем стандартные 80/20

In [7]:
df['strata'] = df['gender'].astype(str) + "_" + df['age'].astype(str)

train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['strata'])

train.drop(columns=['strata'], inplace=True)
test.drop(columns=['strata'], inplace=True)

## "Типичные" значения
Следующие признаки необходимо преобразовать в бинарные по вхождению в "типичный" интервал:
- `entities__mean_usage_MEAS`, 
- `entities__mean_usage_ENUM`,
- `entities__mean_usage_FOREIGN`, 
- `pos__breadth_of_use_pos`,
- `pos__index_of_formality_tuldava`, 
- `entities__mean_usage_SMILE`,
- `entities__mean_usage_DATE`

Диапазоны интервалов определяем по обучающей выборке

In [8]:
typical_features = [ 
    'entities__mean_usage_MEAS', 
    'entities__mean_usage_ENUM',
    'entities__mean_usage_FOREIGN', 
    'pos__breadth_of_use_pos',
    'pos__index_of_formality_tuldava', 
    'entities__mean_usage_SMILE',
    'entities__mean_usage_DATE',
]

def get_typical_range(values, iqr_ratio=0.1):
    result = {}
    
    median = values.median()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    
    result['median'] = median
    
    if iqr == 0:
        unique_values = values.sort_values().unique()
        min_diff = np.min(np.diff(unique_values))
        iqr = min_diff
        
    lower_bound = median - iqr_ratio * iqr
    upper_bound = median + iqr_ratio * iqr
    
    result['lower_bound'] = lower_bound
    result['upper_bound'] = upper_bound
    
    return result

typical_ranges = []
for col in typical_features:
    typical_range = get_typical_range(df[col])
    typical_ranges.append(typical_range)
    
typical_ranges_df = pd.DataFrame(typical_ranges, index=typical_features)
typical_ranges_df

,median,lower_bound,upper_bound
entities__mean_usage_MEAS,0.0000,-0.000005,0.000005
entities__mean_usage_ENUM,0.0000,-0.002222,0.002222
entities__mean_usage_FOREIGN,0.0000,-0.004762,0.004762
pos__breadth_of_use_pos,0.8125,0.806250,0.818750
pos__index_of_formality_tuldava,100.0000,98.449087,101.550913
entities__mean_usage_SMILE,0.0000,-0.000005,0.000005
entities__mean_usage_DATE,0.0000,-0.001010,0.001010


In [9]:
for col in typical_features:
    train[f"{col}_is_typical"] = (train[col].between(typical_ranges_df.loc[col].upper_bound, typical_ranges_df.loc[col].lower_bound)).astype(int)
    train.drop(columns=[col], inplace=True)
    
    test[f"{col}_is_typical"] = (test[col].between(typical_ranges_df.loc[col].upper_bound, typical_ranges_df.loc[col].lower_bound)).astype(int)
    test.drop(columns=[col], inplace=True)

## Сохраняем выборки

In [10]:
train.to_parquet("../data/processed/train.parquet")
test.to_parquet("../data/processed/test.parquet")